# Chapter 1 — Galaxies as Dynamical Systems

This notebook accompanies the first chapter of the lecture notes.
The goal is to turn the order-of-magnitude arguments about timescales and
distribution functions into concrete numerical experiments.

**Contents**
1. Characteristic timescales and the collisionless limit
2. The collisionless boundary
3. Distribution functions — first contact
4. Open exploration — the Coma cluster


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
import astropy.constants as const

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "grid.linestyle": ":",
})

T_HUBBLE = 13.8 * u.Gyr   # approximate age of the Universe


---
## 1. Characteristic timescales and the collisionless limit

For a self-gravitating system of $N$ particles with characteristic size $R$
and velocity dispersion $\sigma$, two timescales dominate the dynamics.

**Crossing time** — the time for a typical particle to cross the system:
$$
t_{\rm cross} = \frac{R}{\sigma}.
$$

**Relaxation time** — the time for cumulative two-body gravitational
encounters to change a particle's velocity by order itself (Chandrasekhar 1942;
Binney & Tremaine 2008, eq. 1.33):
$$
t_{\rm relax} \approx \frac{N}{8\ln N}\,t_{\rm cross}.
$$

When $t_{\rm relax} \gg t_H$ (the Hubble time), two-body encounters are
negligible and the system is **collisionless** — the smooth mean-field
potential governs the dynamics, not individual star-star encounters.


In [ ]:
# Representative stellar systems
# R is a characteristic (half-light) radius; sigma is the 1-D velocity dispersion.
systems = [
    dict(name="Milky Way",      N=2e11,  R=15.0 * u.kpc,   sigma=120.0 * u.km/u.s),
    dict(name="Sculptor dSph",  N=1e7,   R=0.30 * u.kpc,   sigma= 10.0 * u.km/u.s),
    dict(name="47 Tuc (GC)",    N=5e5,   R=5.0  * u.pc,    sigma= 11.0 * u.km/u.s),
    dict(name="Pleiades (OC)",  N=1e3,   R=2.0  * u.pc,    sigma=  0.5 * u.km/u.s),
]

for s in systems:
    s["t_cross"] = (s["R"] / s["sigma"]).to(u.Gyr)
    s["t_relax"] = (s["N"] / (8.0 * np.log(s["N"]))) * s["t_cross"]
    s["ratio"]   = float((s["t_relax"] / T_HUBBLE).decompose())


In [ ]:
# Print a summary table
print(f"{'System':<18} {'N':>8}  {'t_cross':>12}  {'t_relax':>14}  {'t_relax / t_H':>14}")
print("─" * 75)
for s in systems:
    print(
        f"{s['name']:<18} {s['N']:>8.0e}"
        f"  {s['t_cross'].to(u.Myr).value:>9.1f} Myr"
        f"  {s['t_relax'].to(u.Gyr).value:>11.3e} Gyr"
        f"  {s['ratio']:>14.3e}"
    )
print()
print(f"Hubble time  t_H = {T_HUBBLE}")


In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))

names      = [s["name"] for s in systems]
log_ratios = [np.log10(s["ratio"]) for s in systems]
colors     = ["#3a7d44" if r > 0 else "#c0392b" for r in log_ratios]

bars = ax.barh(names, log_ratios, color=colors, alpha=0.75, height=0.55)
ax.axvline(0, color="k", linewidth=1.5, linestyle="--", label=r"$t_{\rm relax} = t_H$")

for bar, val in zip(bars, log_ratios):
    x_label = val + 0.25 if val >= 0 else val - 0.25
    ha = "left" if val >= 0 else "right"
    ax.text(x_label, bar.get_y() + bar.get_height() / 2,
            f"{val:+.1f}", va="center", ha=ha, fontsize=9)

ax.set_xlabel(r"$\log_{10}(t_{\rm relax}\,/\,t_H)$")
ax.set_title("Relaxation time relative to the Hubble time")
ax.legend(fontsize=9)

# Shade regions
xlim = ax.get_xlim()
ax.axvspan(xlim[0], 0,  alpha=0.06, color="#c0392b", zorder=0)
ax.axvspan(0, xlim[1],  alpha=0.06, color="#3a7d44", zorder=0)
ax.set_xlim(xlim)

plt.tight_layout()
plt.show()


> **Exercise 1.1** — Add a system of your own choosing (e.g. a galaxy cluster,
> a nuclear star cluster, or a globular cluster in M31). Look up or estimate
> $R$, $\sigma$, and $N$, add it to the `systems` list, and re-run the cells.


In [ ]:
# your system here


---
## 2. The collisionless boundary

The relaxation time scales as $t_{\rm relax} \propto N / \ln N$, so the
collisional/collisionless divide is primarily a question of particle number.
The next plot shows $t_{\rm relax}/t_H$ as a function of $N$ for fixed
representative $R$ and $\sigma$, with our four systems overlaid.


In [ ]:
N_grid     = np.logspace(2, 14, 800)
R_ref      = 10.0 * u.kpc
sigma_ref  = 100.0 * u.km / u.s
t_cross_ref = (R_ref / sigma_ref).to(u.Gyr).value   # Gyr

t_relax_grid = (N_grid / (8.0 * np.log(N_grid))) * t_cross_ref
ratio_grid   = t_relax_grid / T_HUBBLE.to(u.Gyr).value

fig, ax = plt.subplots(figsize=(8, 5))

ax.loglog(N_grid, ratio_grid, linewidth=2, color="C0",
          label=rf"$R = {R_ref.value:.0f}$ kpc, $\sigma = {sigma_ref.value:.0f}$ km/s")

# Shading
ax.fill_between(N_grid, ratio_grid, 1,
                where=(ratio_grid < 1), alpha=0.12, color="C3")
ax.fill_between(N_grid, ratio_grid, ax.get_ylim()[1] if ax.get_ylim()[1] > 1 else 1e6,
                where=(ratio_grid > 1), alpha=0.10, color="C2")
ax.axhline(1, color="k", linewidth=1.5, linestyle="--",
           label=r"$t_{\rm relax} = t_H$")

# Annotate systems
for s in systems:
    ax.plot(s["N"], s["ratio"], "o", color="k", markersize=7, zorder=5)
    ax.annotate(
        s["name"],
        xy=(s["N"], s["ratio"]),
        xytext=(6, 3),
        textcoords="offset points",
        fontsize=8.5,
    )

ax.set_xlabel("$N$ (number of particles)")
ax.set_ylabel(r"$t_{\rm relax}\,/\,t_H$")
ax.set_title("The collisionless boundary")

# Region labels
ax.text(1e3,  3e-3, "collisional", color="C3", fontsize=10, style="italic")
ax.text(3e8,  30,   "collisionless", color="C2", fontsize=10, style="italic")

ax.legend(fontsize=9, loc="upper left")
ax.set_xlim(N_grid[0], N_grid[-1])
ax.set_ylim(1e-6, 1e10)

plt.tight_layout()
plt.show()


> **Exercise 2.1** — The boundary depends on both $N$ and the ratio $R/\sigma$.
> Add a second curve for a more compact, hotter system
> (e.g. $R = 5$ pc, $\sigma = 10$ km/s, appropriate for a nuclear star cluster)
> and compare where the boundary falls.


---
## 3. Distribution functions — first contact

In the collisionless limit, all the information about the system is encoded
in the **distribution function** $f(\mathbf{x}, \mathbf{v}, t)$, such that

$$
f(\mathbf{x}, \mathbf{v}, t)\,d^3x\,d^3v
$$

is the mass (or number) of stars in the phase-space volume element
$d^3x\,d^3v$ around $(\mathbf{x}, \mathbf{v})$ at time $t$.

The simplest non-trivial DF is the **isotropic Gaussian** (Maxwell–Boltzmann):

$$
f(\mathbf{v}) = \frac{1}{(2\pi\sigma^2)^{3/2}}
\exp\!\left(-\frac{|\mathbf{v}|^2}{2\sigma^2}\right),
$$

where $\sigma$ is the 1-D velocity dispersion.  We work here in velocity
space only (ignoring the spatial dependence for now).

### 3.1 Moments

The moments of $f$ give the familiar macroscopic quantities:
- density: $\rho = \int f\,d^3v$ (here normalised to 1)
- mean velocity: $\langle \mathbf{v} \rangle = \int \mathbf{v}\,f\,d^3v = \mathbf{0}$
- velocity dispersion: $\langle v_x^2 \rangle = \sigma^2$,  $\langle |\mathbf{v}|^2 \rangle = 3\sigma^2$


In [ ]:
rng     = np.random.default_rng(seed=2026)
N_mc    = 300_000          # Monte Carlo sample size
sigma_v = 120.0            # km/s — 1-D velocity dispersion (MW-disk-like)

# Draw from isotropic 3-D Gaussian
vx = rng.normal(0.0, sigma_v, N_mc)
vy = rng.normal(0.0, sigma_v, N_mc)
vz = rng.normal(0.0, sigma_v, N_mc)
speed = np.sqrt(vx**2 + vy**2 + vz**2)

print(f"sigma_v = {sigma_v} km/s\n")
print("Numerically recovered moments:")
print(f"  <vx>              = {vx.mean():+7.3f} km/s     (expected   0.000)")
print(f"  std(vx)           = {vx.std():7.3f} km/s     (expected {sigma_v:.3f})")
print(f"  rms speed         = {np.sqrt(np.mean(speed**2)):7.3f} km/s     "
      f"(expected {sigma_v * np.sqrt(3):.3f}  =  sqrt(3) sigma)")
print(f"  mean speed <|v|>  = {speed.mean():7.3f} km/s     "
      f"(expected {sigma_v * np.sqrt(8.0/np.pi):.3f}  =  sqrt(8/pi) sigma)")


In [ ]:
v_grid = np.linspace(0.0, 6.0 * sigma_v, 600)
v_1d   = np.linspace(-5.0 * sigma_v, 5.0 * sigma_v, 600)

# Theoretical 1-D marginal (integrating out the other two components)
gauss_1d = np.exp(-v_1d**2 / (2.0 * sigma_v**2)) / (sigma_v * np.sqrt(2.0 * np.pi))

# Maxwell-Boltzmann speed distribution: f(|v|) = 4pi * (2pi sigma^2)^{-3/2} * |v|^2 * exp(...)
f_MB = (4.0 * np.pi * (2.0 * np.pi * sigma_v**2)**(-1.5)
        * v_grid**2
        * np.exp(-v_grid**2 / (2.0 * sigma_v**2)))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left panel: 1-D velocity distributions
ax = axes[0]
bins_1d = np.linspace(-5.0 * sigma_v, 5.0 * sigma_v, 80)
for vi, label, color in [(vx, r"$v_x$", "C0"), (vy, r"$v_y$", "C1"), (vz, r"$v_z$", "C2")]:
    ax.hist(vi, bins=bins_1d, density=True, histtype="step",
            color=color, linewidth=1.6, label=label)
ax.plot(v_1d, gauss_1d, "k--", linewidth=1.8, label=r"$\mathcal{N}(0,\sigma^2)$")
ax.set_xlabel("$v$ (km/s)")
ax.set_ylabel("Probability density")
ax.set_title("Component velocity distributions")
ax.legend(fontsize=9)

# Right panel: speed distribution
ax = axes[1]
bins_speed = np.linspace(0.0, 6.0 * sigma_v, 70)
ax.hist(speed, bins=bins_speed, density=True, histtype="step",
        color="C0", linewidth=1.6, label="Monte Carlo")
ax.plot(v_grid, f_MB, "k--", linewidth=1.8, label="Maxwell–Boltzmann")
ax.axvline(sigma_v * np.sqrt(3.0),       color="C3", linestyle=":",
           label=r"$\sqrt{3}\,\sigma$ (rms)")
ax.axvline(sigma_v * np.sqrt(8.0/np.pi), color="C1", linestyle=":",
           label=r"$\sqrt{8/\pi}\,\sigma$ (mean)")
ax.set_xlabel(r"$|\mathbf{v}|$ (km/s)")
ax.set_ylabel("Probability density")
ax.set_title("Speed distribution")
ax.legend(fontsize=8.5)

plt.tight_layout()
plt.show()


### 3.2 The line-of-sight velocity distribution (LOSVD)

Observers measure velocities projected along the line of sight.  If the line
of sight is the $z$-axis, the **observed** LOSVD is the marginal:

$$
\xi(v_{\rm los}) = \int_{-\infty}^{\infty}\!\int_{-\infty}^{\infty}
f(v_x, v_y, v_{\rm los})\,dv_x\,dv_y.
$$

For an isotropic Gaussian, the integral over the two transverse components
leaves a 1-D Gaussian with the same $\sigma$:

$$
\xi(v_{\rm los}) = \frac{1}{\sqrt{2\pi}\,\sigma}
\exp\!\left(-\frac{v_{\rm los}^2}{2\sigma^2}\right).
$$


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

bins_los = np.linspace(-4.5 * sigma_v, 4.5 * sigma_v, 65)
ax.hist(vz, bins=bins_los, density=True,
        histtype="bar", color="C0", alpha=0.30, label="MC sample of $v_z$")
ax.hist(vz, bins=bins_los, density=True,
        histtype="step", color="C0", linewidth=1.6)

v_los = np.linspace(-4.5 * sigma_v, 4.5 * sigma_v, 500)
ax.plot(v_los, np.exp(-v_los**2 / (2.0 * sigma_v**2)) / (sigma_v * np.sqrt(2.0 * np.pi)),
        "k--", linewidth=2.0, label=r"$\xi(v_{\rm los})$ — Gaussian$(0,\sigma^2)$")

ax.set_xlabel(r"$v_{\rm los}$ (km/s)")
ax.set_ylabel("Probability density")
ax.set_title("Line-of-sight velocity distribution")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()


> **Exercise 3.1** — The result above holds because the DF is isotropic.
> Consider an **anisotropic** Gaussian with different dispersions along each axis:
>
> $$f(\mathbf{v}) \propto \exp\!\left(-\frac{v_R^2}{2\sigma_R^2}
> -\frac{v_\phi^2}{2\sigma_\phi^2}
> -\frac{v_z^2}{2\sigma_z^2}\right), \qquad \sigma_R > \sigma_\phi \approx \sigma_z.$$
>
> 1. Sample from such a DF (choose values inspired by the Solar neighbourhood:
>    $\sigma_R \approx 40$, $\sigma_\phi \approx 25$, $\sigma_z \approx 20$ km/s).
> 2. Plot the LOSVD for two lines of sight: one along $R$ and one along $z$.
> 3. What is the shape of each LOSVD, and why are they different?


In [ ]:
# Exercise 3.1 — anisotropic DF


---
## 4. Open exploration — the Coma cluster

The Coma cluster (Abell 1656) is one of the closest rich galaxy clusters.
Its basic parameters are:

| Quantity | Value |
|---|---|
| Total mass $M$ | $\approx 10^{15}\,M_\odot$ |
| Virial radius $R_{200}$ | $\approx 2$ Mpc |
| Velocity dispersion $\sigma$ | $\approx 1000$ km/s |
| Number of bright galaxies $N_{\rm gal}$ | $\approx 1000$ |

Answer the following questions using the tools from this notebook.

1. Compute $t_{\rm cross}$ and $t_{\rm relax}$ for the Coma cluster,
   treating the $\sim 1000$ brightest galaxies as the "particles".
   Is the cluster collisional or collisionless on this basis?

2. The cluster also hosts $N_{\rm sub} \sim 10^4$ dark-matter subhalos.
   How does the answer change if you treat subhalos as the particles?

3. For small galaxy *groups* ($N \sim 50$, $R \sim 500$ kpc,
   $\sigma \sim 300$ km/s), is the assumption of collisionlessness valid?

4. *(Challenge)* The intra-cluster medium (hot X-ray gas) is definitely
   collisional — but its Coulomb relaxation time is set by electron-ion
   collisions, not by the two-body formula above.  Why does the formula
   not apply to a plasma?


In [ ]:
# Open exploration — Coma cluster
